# Introduction to Pickling

---

## Learning objectives

- Learn what serialization and deserialization are
- Learn what "pickling" is in Python
- Review using `with` statements to safely handle file operations
- Pickle and unpickle scikit-learn models in Python

---

## What is pickling?

If you're talking about food, pickling is a method of preserving food for the future. If you're talking about Python, pickling is a method of preserving **objects** for the future, including functions and classes. Since sklearn models are instances of classes, they can be pickled.

To pickle an object, it needs to be **serialized**. Serialization is when we transform an object into byte streams. (Byte streams are collections of bytes. One byte is made up of eight zeros or ones.) To unpickle an object so that it can be used in Python again, it needs to be **deserialized**.

If you've ever saved your progress in a video game, you've already serialized data without knowing it. A save file is your serialized save state. When you load the save, you deserialize the data so you can resume the game right where you were before you quit.

### Some warnings:

Just like you can't open a [Pokemon: Red](https://en.wikipedia.org/wiki/Pok%C3%A9mon_Red_and_Blue) savefile in [Pokemon: Sun](https://en.wikipedia.org/wiki/Pok%C3%A9mon_Sun_and_Moon), you have to unpickle an object in the same version of Python that you pickled it in. 

**Pickle objects can contain malicious code**. Never unpickle an object you don't trust!

## Why pickle?

Pickling makes a lot of sense any time you have a model you want to work with that you don't want to refit. Today, we'll pickle a fitted pipeline so that we can import it into a Streamlit web app, but pickling is useful in many other situations as well.

If you have a model that took twelve hours to fit, you might want to analyze its residuals, work with its coefficients, or make predictions off of it. But without saving it some fashion, you'd need to refit the model every time you restarted your notebook. Pickling the model allows you to load the fitted model _without_ needing to re-run the code where you fit it.

Notes:
- Pickling does **not** compress your model, meaning that some pickled models can end up being fairly large file sizes. Think of K-nearest neighbors, which requires every data point to be stored inside the model (though sklearn _does_ optimize the way the data is stored for speed and efficiency, models can still be large.) 
- Keras has its own [`save` method](https://www.tensorflow.org/guide/keras/save_and_serialize) on models. If you want to save a neural network fit in keras, use that instead of pickle.
- Don't pickle data frames. Export them as csv files instead. Generally, if there's another way to save something, use it.


---

In [1]:
import pandas as pd
import pickle

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

## Pickling a simple datatype

Before we pickle a full model, let's demonstrate pickling on a simple list.

Create a list called `my_vegetables` that contains some strings:

In [2]:
my_vegetables = ['cucumber', 'red pepper', 'onion', 'beet']

### Write the pickled list to disk

Let's review [this link](https://www.pythonforbeginners.com/files/with-statement-in-python) to go over why `with` is such a good tool for file operations.

Let's use `with` to write the list to disk as a `.pkl` file. We'll need to use `open`, pass in a file name, and also tell Python we're **writing** to the file, and writing as **bytes**. The pickle method we'll use is called `dump`.

In [3]:
with open('list_of_vegetables.pkl', 'wb') as pickle_out:  
    pickle.dump(my_vegetables, pickle_out)

### Open the pickled list

Let's use `with` to open the pickled file and save it as a new variable, `list_from_pickle`. Remember to tell Python that we're **reading** from the file, and that we're reading in **bytes**. The pickle method we'll use is called `load`.

In [4]:
with open('list_of_vegetables.pkl', 'rb') as pickle_in:
    list_from_pickle = pickle.load(pickle_in)
    
list_from_pickle

['cucumber', 'red pepper', 'onion', 'beet']

In [5]:
type(list_from_pickle)

list

---

## Pickle a fitted pipeline

Let's start by building a model to determine whether someone writes more like [Edgar Allen Poe](https://en.wikipedia.org/wiki/Edgar_Allan_Poe) or [Jane Austen](https://en.wikipedia.org/wiki/Jane_Austen).

Our end goal will be a fitted pipeline. But before we export our pipeline, we'll need to settle on a model.

### Import data

In [6]:
df = pd.read_csv('../../data/austen_poe.csv').dropna()
df.head(3)

,text,author
0,SENSE AND SENSIBILITY,Jane Austen
1,by Jane Austen,Jane Austen
2,(1811),Jane Austen


In [9]:
# Shape
df.shape

(18034, 2)

In [10]:
# Value counts
df['author'].value_counts(normalize = True)

Jane Austen        0.607353
Edgar Allan Poe    0.392647
Name: author, dtype: float64

In [11]:
# Set up X and y
X = df['text']
y = df['author']

In [12]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

---

## Comparing models

We could use either the TF-IDF vectorizer or the count vectorizer, and one might work better than the other, so let's try both - and let's try both alongside logistic regression and multinomial Naive Bayes.

In [13]:
# Pipe1
pipe = Pipeline([
    ('cv', CountVectorizer()),
    ('lr', LogisticRegressionCV(solver='liblinear'))
])

pipe.fit(X_train, y_train)
pipe.score(X_train, y_train), pipe.score(X_test, y_test)

(0.9911275415896488, 0.9467731204258151)

In [14]:
# Pipe2
pipe2 = Pipeline([
    ('cv', TfidfVectorizer()),
    ('lr', LogisticRegressionCV(solver='liblinear'))
])

pipe2.fit(X_train, y_train)
pipe2.score(X_train, y_train), pipe2.score(X_test, y_test)

(0.994011090573013, 0.9580838323353293)

In [15]:
# Pipe3
pipe3 = Pipeline([
    ('cv', CountVectorizer()),
    ('mnb', MultinomialNB())
])

pipe3.fit(X_train, y_train)
pipe3.score(X_train, y_train), pipe3.score(X_test, y_test)

(0.9510536044362292, 0.9387890884896873)

In [16]:
# Pipe4
pipe4 = Pipeline([
    ('tf', TfidfVectorizer()),
    ('mnb', MultinomialNB())
])

pipe4.fit(X_train, y_train)
pipe4.score(X_train, y_train), pipe4.score(X_test, y_test)

(0.9191127541589649, 0.8942115768463074)

----

My best model:


---

## Pickle a fitted pipeline


### Export the fitted pipeline to `models` as `author_pipe.pkl`

This time, let's export to the `models` folder in this repository.

> Why bother? First, if you'll have lots of serialized objects, or if your serialized objects take up lots of disk space, you might not want to add and commit them to Github. Keeping them all in the same folder makes it easier to stay organized and not commit them. Second, it's also just a good way to organize a repository!

Just like before, we'll use a `with` statement:

In [17]:
with open('../../models/author_pipe.pkl', 'wb') as pickle_out:
    pickle_out = pickle.dump(pipe2, pickle_out)

Next we'll try to un-pickle in the `02-read_a_pickle` notebook.